# Phase 5a - all 5 datasets -> device cues (10 base + 9 extra) for the Phase 5 experiments

| Dataset | Source on Kaggle | Cues |
|---|---|---|
| mat80 (80 MHz people counting) | `wizard1803/wifi-csi` `Dataset/` | Phase 1 -> **causal** (device) cues, 5-s windows every 2.5 s, 16 views |
| CountFi | `wizard1803/wifi-csi` `CountFi/` | same, 4 views x 2 receivers |
| WiCount (ESP32-S3) | `wizard1803/wifi-csi` `WiCount/` | same, 1 view |
| WiMANS 5 GHz | `shuokanghuang/wimans` | clip cues (3-s clips, clip-mean normalised), 9 links |
| CSI-Bench Localization | `guozhenjennzhu/csi-bench` | clip cues (5-s clips), 3 devices |

Base cues = the 10 the ESP32 model uses (`esp32_counter.py`, written into this kernel unchanged).
Extra cues = the 9 "how many separate movers" cues from the WiMANS / CSI-Bench notebooks.

Output in `/kaggle/working/`: `streams_features.parquet`, `wimans_clip_features.parquet`,
`csibench_clip_features.parquet`. CPU only, ~1-2 h.

In [ ]:
# 1. Setup: find the three local datasets inside wizard1803/wifi-csi
import glob
import json
import os
import shutil
import struct
import sys
import time
import zlib
from multiprocessing import get_context
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))


def one_dir(test, what):
    hits = find_dirs(test, max_depth=8)
    if not hits:
        raise FileNotFoundError(f"{what} not found under /kaggle/input - is wizard1803/wifi-csi attached?")
    return hits[0]


MAT80_ROOT = os.path.dirname(one_dir(lambda d, f: os.path.basename(d) == "PC-1a", "Dataset/PC-1a"))
COUNTFI_ROOT = os.path.dirname(one_dir(lambda d, f: os.path.basename(d) == "48"
                                       and "CountFi" in d, "CountFi/.../48"))
WICOUNT_DATA = one_dir(lambda d, f: os.path.basename(d) == "data" and "WiCount" in d
                       and os.path.isdir(os.path.join(d, "0")), "WiCount/data")
print("mat80  :", MAT80_ROOT)
print("CountFi:", COUNTFI_ROOT)
print("WiCount:", WICOUNT_DATA)

OUT_ROOT = "/kaggle/working/shared_format"      # Phase 1 output (temporary)
os.makedirs(OUT_ROOT, exist_ok=True)

# ---- shared-format settings (identical to the laptop Phase 1 and to esp32_counter.py) ----
FS = 50
WIN_S = 5.0
HOP_S = 2.5
MAX_GAP_BINS = 3
MAX_MISSING = 0.20
N_TONES = 52
N_WORKERS = os.cpu_count()
print("cpus:", N_WORKERS)

In [ ]:
# 3. Shared-format helpers: sub-bands, 50 Hz grid, normalisation, count cues

# ---------------------------------------------------------------
# 20 MHz sub-bands of an 80 MHz channel (256-point FFT, natural order)
# ---------------------------------------------------------------
# Sub-band s covers tone indices -128+64s ... -65+64s. We keep 52 tones,
# skipping 6 at each edge. This automatically avoids DC (bins 0, 1, 255)
# and the guard band (bins 123-133) that carry no channel data.
def subband_bins(s):
    start = -128 + 64 * s
    tones = np.arange(start + 6, start + 6 + N_TONES)
    return np.mod(tones, 256)


SUBBANDS = [subband_bins(s) for s in range(4)]


# ---------------------------------------------------------------
# Normalisation
# ---------------------------------------------------------------
def gain_normalise(A):
    """Per packet: divide by the median over tones (removes AGC jumps)."""
    return A / (np.median(A, axis=1, keepdims=True) + 1e-6)


def recording_normalise(X):
    """Per recording + receiver: each tone relative to its mean -> fluctuation around 0."""
    mu = np.nanmean(X, axis=0, keepdims=True)
    dead = mu < 1e-3 * np.nanmedian(mu)                    # null tones, if any
    Y = X / (mu + 1e-6) - 1.0
    Y[:, dead[0]] = 0.0
    return Y


# ---------------------------------------------------------------
# Resampling onto the 50 Hz grid
# ---------------------------------------------------------------
def resample_to_grid(t, A):
    """t: packet times (s, sorted), A: (packets, tones).
    Returns X (bins, tones) with NaN rows for empty bins, and the valid mask."""
    idx = np.floor((t - t[0]) * FS).astype(np.int64)
    n_bins = int(idx[-1]) + 1

    uniq, first = np.unique(idx, return_index=True)        # idx is sorted
    sums = np.add.reduceat(A.astype(np.float64), first, axis=0)
    counts = np.diff(np.append(first, len(idx)))

    X = np.full((n_bins, A.shape[1]), np.nan, dtype=np.float32)
    X[uniq] = sums / counts[:, None]
    valid = np.zeros(n_bins, dtype=bool)
    valid[uniq] = True
    return X, valid


def fill_short_gaps(X, valid):
    """Linear interpolation over gaps of at most MAX_GAP_BINS; longer gaps stay NaN."""
    X, valid = X.copy(), valid.copy()
    missing = np.flatnonzero(~valid)
    if len(missing) == 0:
        return X, valid

    runs = np.split(missing, np.flatnonzero(np.diff(missing) > 1) + 1)
    for run in runs:
        left, right = run[0] - 1, run[-1] + 1
        if len(run) <= MAX_GAP_BINS and left >= 0 and right < len(X):
            w = ((run - left) / (right - left))[:, None]
            X[run] = X[left] * (1 - w) + X[right] * w
            valid[run] = True
    return X, valid


def to_shared_series(t, A):
    """Raw amplitude of ONE view -> normalised 50 Hz series (bins, 52) + valid mask."""
    A = gain_normalise(A.astype(np.float32))
    X, valid = resample_to_grid(t, A)
    X, valid = fill_short_gaps(X, valid)
    return recording_normalise(X), valid


# ---------------------------------------------------------------
# Count cues per window
# ---------------------------------------------------------------
FEATURE_NAMES = [
    "cv_median",       # median over tones of the fluctuation std
    "cv_p90",          # 90th percentile of the same
    "cv_p10",          # 10th percentile of the same
    "diff_energy",     # mean |x(t) - x(t-1)|  (motion energy)
    "pc1_frac",        # share of variance in the first principal component
    "n_pc_90",         # components needed for 90 % of the variance
    "tone_corr",       # mean correlation between tones
    "low_freq_frac",   # temporal power below 2 Hz (slow body motion)
    "high_freq_frac",  # temporal power above 5 Hz
    "mean_std",        # std of the tone-averaged series
]


def window_features(W):
    """W: (bins, 52) normalised window. Missing bins count as 'no change' (0)."""
    W = np.nan_to_num(W.astype(np.float32))
    sd = W.std(axis=0)
    diff_energy = np.abs(np.diff(W, axis=0)).mean()

    Xc = W - W.mean(axis=0)
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    n90 = np.searchsorted(np.cumsum(frac), 0.90) + 1

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(Xc.T)
    tone_corr = np.nanmean(C[np.triu_indices_from(C, k=1)])

    x = Xc.mean(axis=1)
    P = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / FS)
    p_total = P[f > 0].sum() + 1e-12
    low = P[(f > 0) & (f < 2.0)].sum() / p_total
    high = P[f > 5.0].sum() / p_total

    return np.array([np.median(sd), np.percentile(sd, 90), np.percentile(sd, 10),
                     diff_energy, frac[0], n90, tone_corr, low, high, x.std()],
                    dtype=np.float32)


# ---------------------------------------------------------------
# Windowing + saving one recording
# ---------------------------------------------------------------
def save_recording(rec):
    """rec: dict with series (views, bins, 52), valid (bins,), view_names, metadata."""
    S, valid = rec["series"], rec["valid"]
    win, hop = int(WIN_S * FS), int(HOP_S * FS)

    starts = [a for a in range(0, S.shape[1] - win + 1, hop)
              if valid[a:a + win].mean() >= 1 - MAX_MISSING]

    F = np.zeros((S.shape[0], len(starts), len(FEATURE_NAMES)), dtype=np.float32)
    for v in range(S.shape[0]):
        for j, a in enumerate(starts):
            F[v, j] = window_features(S[v, a:a + win])

    meta = {k: rec[k] for k in ["dataset", "recording", "group", "setup", "receiver", "count"]}
    meta["seconds"] = round(S.shape[1] / FS, 1)
    meta["missing_frac"] = round(float(1 - valid.mean()), 4)
    meta["n_windows"] = len(starts)

    out_dir = os.path.join(OUT_ROOT, rec["dataset"])
    os.makedirs(out_dir, exist_ok=True)
    np.savez_compressed(os.path.join(out_dir, rec["recording"] + ".npz"),
                        series=S.astype(np.float16), valid=valid,
                        starts=np.array(starts, dtype=np.int32), feats=F,
                        view_names=np.array(rec["view_names"]), meta=json.dumps(meta))
    return meta


def already_done(dataset, recording):
    return os.path.exists(os.path.join(OUT_ROOT, dataset, recording + ".npz"))


print("sub-band bins:", [(int(b[0]), int(b[-1])) for b in SUBBANDS])

In [ ]:
# 4. 80 MHz set (Dataset/PC-*a/*.mat): 4 antennas x 4 sub-bands = 16 views per recording
MAT80_RATE = 173          # packets/s per antenna (published; the files have no timestamps)
MI_TYPES = {1: "i1", 2: "u1", 3: "<i2", 4: "<u2", 5: "<i4", 6: "<u4", 7: "<f4", 9: "<f8"}


# ---------------------------------------------------------------
# Streaming MAT-v5 reader (column by column, low RAM)
# ---------------------------------------------------------------
class ZStream:
    def __init__(self, f, comp_len):
        self.f, self.left = f, comp_len
        self.d, self.buf = zlib.decompressobj(), bytearray()

    def read(self, n):
        while len(self.buf) < n:
            if self.left <= 0:
                self.buf += self.d.flush()
                break
            chunk = self.f.read(min(4 << 20, self.left))
            self.left -= len(chunk)
            self.buf += self.d.decompress(chunk)
        out = bytes(self.buf[:n])
        del self.buf[:n]
        return out


def read_tag(src):
    raw = src.read(8)
    t, n = struct.unpack("<II", raw)
    if t >> 16:                                   # small data element
        return t & 0xFFFF, t >> 16, raw[4:4 + (t >> 16)], True
    return t, n, None, False


def read_element(src):
    t, n, small, is_small = read_tag(src)
    if is_small:
        return t, small
    data = src.read(n)
    if n % 8:
        src.read(8 - n % 8)
    return t, data


def load_amplitude(path):
    """Return amplitude (rows, 256) float16, or None if the file holds no data."""
    with open(path, "rb") as f:
        f.read(128)
        _, n = struct.unpack("<II", f.read(8))
        src = ZStream(f, n)
        read_tag(src)                             # miMATRIX
        read_element(src)                         # flags
        _, dims = read_element(src)
        nrows, ncols = struct.unpack("<ii", dims[:8])
        read_element(src)                         # variable name
        if nrows == 0:
            return None

        t, nbytes, _, _ = read_tag(src)           # real part
        dt = np.dtype(MI_TYPES[t])
        R = np.empty((nrows, ncols), dtype=np.int16)
        for c in range(ncols):
            R[:, c] = np.frombuffer(src.read(nrows * dt.itemsize), dtype=dt)
        if nbytes % 8:
            src.read(8 - nbytes % 8)

        t, nbytes, _, _ = read_tag(src)           # imaginary part -> amplitude
        dt = np.dtype(MI_TYPES[t])
        A = np.empty((nrows, ncols), dtype=np.float16)
        for c in range(ncols):
            im = np.frombuffer(src.read(nrows * dt.itemsize), dtype=dt).astype(np.float32)
            A[:, c] = np.hypot(R[:, c].astype(np.float32), im)
    return A


def process_mat80(path):
    name = Path(path).stem                         # PC1a_n02
    setup = name.split("_")[0]
    count = int(name.split("_n")[1])
    try:
        A = load_amplitude(path)
    except struct.error:                           # header only, no data stream
        A = None
    if A is None:
        return None

    n = (len(A) // 4) * 4                          # rows cycle over the 4 antennas
    series, views, valid_all = [], [], None
    for ant in range(4):
        A_ant = A[ant:n:4]
        t = np.arange(len(A_ant)) / MAT80_RATE
        for s, bins in enumerate(SUBBANDS):
            X, valid = to_shared_series(t, A_ant[:, bins])
            series.append(X)
            views.append(f"ant{ant}_sb{s}")
            valid_all = valid if valid_all is None else (valid_all & valid)

    return dict(dataset="mat80", recording=name, group=setup, setup=setup, receiver="rx",
                count=count, series=np.stack(series), valid=valid_all, view_names=views)


mat_files = sorted(glob.glob(os.path.join(MAT80_ROOT, "PC-*", "*.mat")))
print(len(mat_files), "MAT files")
t0 = time.time()
for path in mat_files:
    name = Path(path).stem
    if already_done("mat80", name):
        continue
    rec = process_mat80(path)
    if rec is None:
        print(f"{name:12s} EMPTY - skipped")
        continue
    meta = save_recording(rec)
    print(f"{name:12s} count={meta['count']:2d}  {meta['seconds']:6.1f} s  "
          f"windows={meta['n_windows']:3d}  missing={meta['missing_frac']:.3f}  "
          f"[{time.time() - t0:.0f} s]")

In [ ]:
# 5. CountFi (CountFi/48/48_<rx>_<scenario>/<count>/*.pcap): QoS-data frames only, 4 sub-bands
# One recording = one (scenario, receiver, count) block.
# Each .pcap is a ~1 s burst (1000 packets) followed by ~1.2 s of nothing before the next file,
# so on real time only ~40 % of the 50 Hz bins would have data and every 5 s window would be dropped.
# The bursts are therefore STITCHED back to back (dead time between files removed).
# Split unit later = scenario (head and end of the same scenario always stay together).
ETH, IP, UDP, NEX = 14, 20, 8, 18


def read_qos(path):
    """Return (times s, amplitude (packets, 256)) of the QoS-data frames, or None."""
    data = open(path, "rb").read()
    off, ts, payloads = 24, [], []
    while off + 16 <= len(data):
        sec, usec, incl, _ = struct.unpack_from("<IIII", data, off)
        off += 16
        pkt = data[off:off + incl]
        off += incl
        p = pkt[ETH + IP + UDP:]
        if p[:2] != b"\x11\x11" or len(p) != NEX + 1024 or p[3] != 0x88:
            continue                               # not a Nexmon QoS-data CSI frame
        ts.append(sec + usec * 1e-6)
        payloads.append(p[NEX:])
    if not payloads:
        return None
    raw = np.frombuffer(b"".join(payloads), dtype="<i2").reshape(len(payloads), 512)
    A = np.hypot(raw[:, 0::2].astype(np.float32), raw[:, 1::2].astype(np.float32))
    return np.array(ts), A


def countfi_blocks(root):
    blocks = {}
    for path in glob.glob(os.path.join(root, "**", "*.pcap"), recursive=True):
        p = Path(path)
        parts = p.parent.parent.name.split("_")    # 48_end_mm_v2 -> ['48','end','mm','v2']
        receiver, scenario = parts[1], "_".join(parts[2:])
        count = int(p.parent.name)
        blocks.setdefault((scenario, receiver, count), []).append(path)
    return blocks


def process_countfi(key, paths):
    scenario, receiver, count = key

    # read every 1 s burst, each sorted by its own timestamps
    bursts = []
    for path in paths:
        r = read_qos(path)
        if r is not None:
            order = np.argsort(r[0], kind="stable")
            bursts.append((r[0][order], r[1][order]))
    if not bursts:
        return None

    # stitch: bursts in time order, each one starting one bin after the previous one ends
    bursts.sort(key=lambda b: b[0][0])
    times, amps, offset = [], [], 0.0
    for t_burst, A_burst in bursts:
        t_local = t_burst - t_burst[0]
        times.append(offset + t_local)
        amps.append(A_burst)
        offset += t_local[-1] + 1.0 / FS

    t = np.concatenate(times)
    A = np.concatenate(amps)

    series, views, valid_all = [], [], None
    for s, bins in enumerate(SUBBANDS):
        X, valid = to_shared_series(t, A[:, bins])
        series.append(X)
        views.append(f"sb{s}")
        valid_all = valid if valid_all is None else (valid_all & valid)

    return dict(dataset="countfi", recording=f"{scenario}_{receiver}_n{count:02d}",
                group=scenario, setup=scenario, receiver=receiver, count=count,
                series=np.stack(series), valid=valid_all, view_names=views)


blocks = countfi_blocks(COUNTFI_ROOT)
print(len(blocks), "CountFi blocks,", sum(len(v) for v in blocks.values()), "pcap files")
t0 = time.time()
for key in sorted(blocks):
    scenario, receiver, count = key
    name = f"{scenario}_{receiver}_n{count:02d}"
    if already_done("countfi", name):
        with np.load(os.path.join(OUT_ROOT, "countfi", name + ".npz")) as old:
            n_old = json.loads(str(old["meta"]))["n_windows"]
        if n_old > 0:
            continue                               # done; redo only files saved before stitching (0 windows)
    rec = process_countfi(key, blocks[key])
    if rec is None:
        print(f"{name:18s} no QoS frames - skipped")
        continue
    meta = save_recording(rec)
    print(f"{name:18s} {meta['seconds']:6.1f} s  windows={meta['n_windows']:3d}  "
          f"missing={meta['missing_frac']:.3f}  [{time.time() - t0:.0f} s]")

In [ ]:
# 6. WiCount (WiCount/data/<count>/*.csv): ESP32-S3, 52 tones already, 1 view
# Split unit later = recording (each file is its own 1-minute session).
WRAP = 2 ** 32                                     # ESP32 local_timestamp is a 32-bit microsecond counter


def process_wicount(path):
    df = pd.read_csv(path)
    rows, keep_t = [], []
    for ts, txt in zip(df["local_timestamp"], df["data"]):
        try:
            vals = json.loads(txt)
        except (TypeError, ValueError):
            continue
        if len(vals) == 2 * N_TONES:
            rows.append(vals)
            keep_t.append(ts)
    if len(rows) < FS:
        return None

    iq = np.array(rows, dtype=np.float32)
    A = np.hypot(iq[:, 0::2], iq[:, 1::2])         # amplitude is the same for I/Q or Q/I order

    dt = np.diff(np.array(keep_t, dtype=np.int64))
    dt[dt < 0] += WRAP                             # counter wrap-around
    t = np.concatenate([[0], np.cumsum(dt)]) * 1e-6
    order = np.argsort(t, kind="stable")

    X, valid = to_shared_series(t[order], A[order])
    name = Path(path).stem
    return dict(dataset="wicount", recording=name, group=name, setup="room", receiver="esp32s3",
                count=int(Path(path).parent.name), series=X[None], valid=valid,
                view_names=["esp32"])


csv_files = sorted(glob.glob(os.path.join(WICOUNT_DATA, "*", "*.csv")))
print(len(csv_files), "WiCount files")
for path in csv_files:
    name = Path(path).stem
    if already_done("wicount", name):
        continue
    rec = process_wicount(path)
    if rec is None:
        print(f"{name} too few valid rows - skipped")
        continue
    meta = save_recording(rec)
    print(f"{name}  count={meta['count']}  {meta['seconds']:5.1f} s  "
          f"windows={meta['n_windows']:3d}  missing={meta['missing_frac']:.3f}")

In [ ]:
%%writefile /kaggle/working/esp32_counter.py
"""ESP32-S3 people counter - reference implementation of the final model.

Everything the firmware has to do, written in Python, packet by packet:

    raw CSI amplitude (52 tones)
      -> per-packet gain normalisation (divide by the median over tones)
      -> 50 Hz bins (packets averaged per 20 ms), gaps up to 60 ms interpolated
      -> causal baseline per tone (EMA, 30 s) -> relative fluctuation z = x / baseline - 1
      -> 5 s windows every 2.5 s (first 10 s skipped while the baseline settles)
      -> 10 count features
      -> log of the magnitude features, minus the empty-room offset (if calibrated)
      -> tree model (LightGBM regression, exported to plain arrays / C header)
      -> clip to the trained count range -> median of the last 30 s -> people count

The batch functions (used for training) and StreamingCounter (what the device runs)
share the same code, and test_esp32_counter.py checks that they give the same numbers.
Nothing here looks at future packets.
"""

import json
import math
import os
import time
from collections import deque

import numpy as np
from scipy.signal import lfilter


# ===============================================================
# Settings (identical on the ESP32)
# ===============================================================
FS = 50                  # sample rate of the shared grid (Hz)
WIN_S = 5.0              # window length (s)
HOP_S = 2.5              # window hop (s)
WIN = int(WIN_S * FS)    # 250 bins
HOP = int(HOP_S * FS)    # 125 bins
N_TONES = 52

MAX_GAP_BINS = 3         # gaps up to 3 bins (60 ms) are interpolated
MAX_MISSING = 0.20       # windows with more missing bins than this are skipped

TAU_S = 30.0             # time constant of the causal per-tone baseline
ALPHA = 1.0 / (TAU_S * FS)
WARMUP_S = 10.0          # no output for the first 10 s (baseline settling)
WARMUP_BINS = int(WARMUP_S * FS)

CAL_S = 20.0             # empty-room calibration after the warm-up
N_CAL_WINDOWS = int((CAL_S - WIN_S) / HOP_S) + 1     # 7 windows

SMOOTH_S = 30.0          # output = median of the last 30 s of window estimates
SMOOTH_N = int(SMOOTH_S / HOP_S)                    # 12 windows

EPS = 1e-6

FEATURE_NAMES = [
    "cv_median",       # median over tones of the fluctuation std
    "cv_p90",          # 90th percentile of the same
    "cv_p10",          # 10th percentile of the same
    "diff_energy",     # mean |x(t) - x(t-1)|  (motion energy)
    "pc1_frac",        # share of variance in the first principal component
    "n_pc_90",         # components needed for 90 % of the variance
    "tone_corr",       # mean correlation between tones
    "low_freq_frac",   # temporal power below 2 Hz (slow body motion)
    "high_freq_frac",  # temporal power above 5 Hz
    "mean_std",        # std of the tone-averaged series
]

# magnitude features go through log() so the empty-room offset acts as a ratio
LOG_FEATURES = ["cv_median", "cv_p90", "cv_p10", "diff_energy", "mean_std"]
LOG_IDX = [FEATURE_NAMES.index(name) for name in LOG_FEATURES]


def settings_dict():
    return {"fs_hz": FS, "win_s": WIN_S, "hop_s": HOP_S, "n_tones": N_TONES,
            "max_gap_bins": MAX_GAP_BINS, "max_missing": MAX_MISSING,
            "tau_s": TAU_S, "warmup_s": WARMUP_S, "cal_s": CAL_S,
            "n_cal_windows": N_CAL_WINDOWS, "smooth_s": SMOOTH_S, "smooth_n": SMOOTH_N,
            "feature_names": FEATURE_NAMES, "log_features": LOG_FEATURES}


# ===============================================================
# Batch preprocessing (training)
# ===============================================================
def gain_normalise(A):
    """Per packet: divide by the median over tones (removes AGC jumps)."""
    return A / (np.median(A, axis=1, keepdims=True) + EPS)


def resample_to_grid(t, A):
    """t: packet times (s, sorted), A: (packets, tones).
    Returns X (bins, tones) with NaN rows for empty bins, and the valid mask."""
    idx = np.floor((t - t[0]) * FS).astype(np.int64)
    n_bins = int(idx[-1]) + 1

    uniq, first = np.unique(idx, return_index=True)        # idx is sorted
    sums = np.add.reduceat(A.astype(np.float64), first, axis=0)
    counts = np.diff(np.append(first, len(idx)))

    X = np.full((n_bins, A.shape[1]), np.nan, dtype=np.float32)
    X[uniq] = sums / counts[:, None]
    valid = np.zeros(n_bins, dtype=bool)
    valid[uniq] = True
    return X, valid


def fill_short_gaps(X, valid):
    """Linear interpolation over gaps of at most MAX_GAP_BINS; longer gaps stay NaN."""
    X, valid = X.copy(), valid.copy()
    missing = np.flatnonzero(~valid)
    if len(missing) == 0:
        return X, valid

    runs = np.split(missing, np.flatnonzero(np.diff(missing) > 1) + 1)
    for run in runs:
        left, right = run[0] - 1, run[-1] + 1
        if len(run) <= MAX_GAP_BINS and left >= 0 and right < len(X):
            w = ((run - left) / (right - left))[:, None]
            X[run] = X[left] * (1 - w) + X[right] * w
            valid[run] = True
    return X, valid


def causal_normalise(X, valid):
    """Each tone relative to its own past (EMA baseline): z = x / baseline - 1.

    The baseline only moves on valid bins and starts at the first valid value.
    Because it is linear, any constant per-tone scale of X cancels out, so the
    Phase 1 series (already divided by the recording mean) give the same z."""
    Z = np.full(X.shape, np.nan, dtype=np.float32)
    rows = np.flatnonzero(valid)
    if len(rows) == 0:
        return Z

    xs = X[rows].astype(np.float64)
    zi = (1.0 - ALPHA) * xs[:1]                             # first output = first value
    baseline, _ = lfilter([ALPHA], [1.0, -(1.0 - ALPHA)], xs, axis=0, zi=zi)

    with np.errstate(invalid="ignore", divide="ignore"):
        z = np.where(baseline > EPS, xs / baseline - 1.0, 0.0)
    Z[rows] = z
    return Z


def window_starts(valid):
    """Window start bins: every HOP, after the warm-up, with enough valid bins."""
    n = len(valid)
    return [a for a in range(0, n - WIN + 1, HOP)
            if a >= WARMUP_BINS and valid[a:a + WIN].mean() >= 1 - MAX_MISSING]


# ===============================================================
# Count features (same code as Phase 1)
# ===============================================================
def window_features(W):
    """W: (bins, 52) normalised window. Missing bins count as 'no change' (0)."""
    W = np.nan_to_num(W.astype(np.float32))
    sd = W.std(axis=0)
    diff_energy = np.abs(np.diff(W, axis=0)).mean()

    Xc = W - W.mean(axis=0)
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    n90 = np.searchsorted(np.cumsum(frac), 0.90) + 1

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(Xc.T)
    tone_corr = np.nanmean(C[np.triu_indices_from(C, k=1)])

    x = Xc.mean(axis=1)
    P = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / FS)
    p_total = P[f > 0].sum() + 1e-12
    low = P[(f > 0) & (f < 2.0)].sum() / p_total
    high = P[f > 5.0].sum() / p_total

    return np.array([np.median(sd), np.percentile(sd, 90), np.percentile(sd, 10),
                     diff_energy, frac[0], n90, tone_corr, low, high, x.std()],
                    dtype=np.float32)


def features_from_series(X, valid):
    """X: (bins, 52) amplitude on the 50 Hz grid (any per-tone scale).
    Returns window start bins and features (windows, 10)."""
    Z = causal_normalise(X, valid)
    starts = window_starts(valid)
    F = np.zeros((len(starts), len(FEATURE_NAMES)), dtype=np.float32)
    for j, a in enumerate(starts):
        F[j] = window_features(Z[a:a + WIN])
    return np.array(starts, dtype=np.int32), F


def features_from_raw(t, A):
    """Raw packets (times s, amplitude (packets, 52)) -> window starts + features."""
    order = np.argsort(t, kind="stable")
    t, A = np.asarray(t, dtype=np.float64)[order], np.asarray(A)[order]
    X, valid = resample_to_grid(t, gain_normalise(A.astype(np.float32)))
    X, valid = fill_short_gaps(X, valid)
    return features_from_series(X, valid)


# ===============================================================
# Model input, calibration, smoothing
# ===============================================================
def prepare_features(F, offset=None):
    """Raw features -> model input: NaN -> 0, log of magnitude features, minus offset."""
    T = np.nan_to_num(np.asarray(F, dtype=np.float64), nan=0.0)
    T = np.atleast_2d(T).copy()
    T[:, LOG_IDX] = np.log(np.maximum(T[:, LOG_IDX], 0.0) + EPS)
    if offset is not None:
        T = T - np.asarray(offset, dtype=np.float64)
    return T.astype(np.float32)


def calibration_offset(F_cal):
    """Empty-room offset = mean model input over the calibration windows."""
    return prepare_features(F_cal).astype(np.float64).mean(axis=0)


def smooth_predictions(pred):
    """Causal rolling median over the last SMOOTH_N window estimates."""
    pred = np.asarray(pred, dtype=np.float64)
    out = np.empty_like(pred)
    for i in range(len(pred)):
        out[i] = np.median(pred[max(0, i - SMOOTH_N + 1):i + 1])
    return out


def to_count(pred, max_count):
    return np.clip(np.rint(pred), 0, max_count).astype(int)


# ===============================================================
# Tree model: LightGBM -> flat arrays (Python, JSON, C header)
# ===============================================================
def _float32_at_most(value):
    """Largest float32 <= value, so 'x <= thr' in float32 equals 'x <= value' in double."""
    thr = np.float32(value)
    if float(thr) > value:
        thr = np.nextafter(thr, np.float32(-np.inf))
    return float(thr)


class TreeModel:
    """Sum of regression trees stored as flat node arrays.

    Node i: feature[i] = -1 for a leaf (value[i] = leaf output), otherwise go to
    left[i] if x[feature[i]] <= value[i] (threshold) else right[i]."""

    def __init__(self, roots, feature, value, left, right, max_count, n_features):
        self.roots = np.asarray(roots, dtype=np.int32)
        self.feature = np.asarray(feature, dtype=np.int32)
        self.value = np.asarray(value, dtype=np.float32)
        self.left = np.asarray(left, dtype=np.int32)
        self.right = np.asarray(right, dtype=np.int32)
        self.max_count = int(max_count)
        self.n_features = int(n_features)

    # ---------------------------------------------------------------
    # Build / save / load
    # ---------------------------------------------------------------
    @classmethod
    def from_lightgbm(cls, booster, max_count):
        dump = booster.dump_model()
        feature, value, left, right, roots = [], [], [], [], []

        def add(node):
            i = len(feature)
            feature.append(-1)
            value.append(0.0)
            left.append(-1)
            right.append(-1)

            if "leaf_value" in node:
                value[i] = float(np.float32(node["leaf_value"]))
                return i

            if node["decision_type"] != "<=":
                raise ValueError(f"unsupported split type {node['decision_type']}")
            if node.get("missing_type", "None") not in ("None", "NaN"):
                raise ValueError(f"unsupported missing type {node['missing_type']}")

            feature[i] = int(node["split_feature"])
            value[i] = _float32_at_most(float(node["threshold"]))
            left[i] = add(node["left_child"])
            right[i] = add(node["right_child"])
            return i

        for tree in dump["tree_info"]:
            roots.append(add(tree["tree_structure"]))

        return cls(roots, feature, value, left, right, max_count,
                   n_features=dump["max_feature_idx"] + 1)

    def to_dict(self):
        return {"roots": self.roots.tolist(), "feature": self.feature.tolist(),
                "value": [float(v) for v in self.value], "left": self.left.tolist(),
                "right": self.right.tolist(), "max_count": self.max_count,
                "n_features": self.n_features}

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.to_dict(), f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            d = json.load(f)
        return cls(d["roots"], d["feature"], d["value"], d["left"], d["right"],
                   d["max_count"], d["n_features"])

    # ---------------------------------------------------------------
    # Inference (float32, same order of operations as the C code)
    # ---------------------------------------------------------------
    def predict(self, X):
        X = np.atleast_2d(np.asarray(X, dtype=np.float32))
        if np.isnan(X).any():
            raise ValueError("model input contains NaN - use prepare_features()")

        rows = np.arange(len(X))
        total = np.zeros(len(X), dtype=np.float32)
        for root in self.roots:
            node = np.full(len(X), root, dtype=np.int32)
            while True:
                f = self.feature[node]
                split = f >= 0
                if not split.any():
                    break
                go_left = X[rows, np.maximum(f, 0)] <= self.value[node]
                node = np.where(split, np.where(go_left, self.left[node], self.right[node]), node)
            total += self.value[node]
        return total

    @property
    def n_nodes(self):
        return len(self.feature)

    # ---------------------------------------------------------------
    # C header for the ESP32 firmware
    # ---------------------------------------------------------------
    def write_c_header(self, path, note=""):
        idx_type = "int16_t" if self.n_nodes < 32767 else "int32_t"

        def array(ctype, name, values, fmt):
            body = ", ".join(fmt(v) for v in values)
            lines, line = [], "    "
            for item in body.split(", "):
                if len(line) + len(item) > 96:
                    lines.append(line.rstrip())
                    line = "    "
                line += item + ", "
            lines.append(line.rstrip().rstrip(","))
            return f"static const {ctype} {name}[{len(values)}] = {{\n" + "\n".join(lines) + "\n};\n"

        log_flags = [1 if name in LOG_FEATURES else 0 for name in FEATURE_NAMES]
        float_fmt = lambda v: f"{float(v):.9g}f"
        int_fmt = lambda v: str(int(v))

        text = [
            "/* ESP32-S3 people counter - generated by esp32_counter.py. Do not edit. */",
            f"/* {note} */" if note else "",
            "#ifndef CC_COUNTER_MODEL_H",
            "#define CC_COUNTER_MODEL_H",
            "",
            "#include <math.h>",
            "#include <stdint.h>",
            "",
            "/* ---- preprocessing settings (must match esp32_counter.py) ---- */",
            f"#define CC_FS_HZ          {FS}",
            f"#define CC_N_TONES        {N_TONES}",
            f"#define CC_WIN_BINS       {WIN}",
            f"#define CC_HOP_BINS       {HOP}",
            f"#define CC_MAX_GAP_BINS   {MAX_GAP_BINS}",
            f"#define CC_MAX_MISSING    {MAX_MISSING}f",
            f"#define CC_EMA_ALPHA      {ALPHA:.9g}f",
            f"#define CC_WARMUP_BINS    {WARMUP_BINS}",
            f"#define CC_N_CAL_WINDOWS  {N_CAL_WINDOWS}",
            f"#define CC_SMOOTH_N       {SMOOTH_N}",
            "",
            "/* ---- model ---- */",
            f"#define CC_N_FEATURES     {self.n_features}",
            f"#define CC_N_TREES        {len(self.roots)}",
            f"#define CC_N_NODES        {self.n_nodes}",
            f"#define CC_MAX_COUNT      {self.max_count}",
            "",
            "/* features, in order: " + ", ".join(FEATURE_NAMES) + " */",
            array("uint8_t", "cc_log_feature", log_flags, int_fmt),
            array(idx_type, "cc_roots", self.roots, int_fmt),
            array("int8_t", "cc_feature", self.feature, int_fmt),
            array("float", "cc_value", self.value, float_fmt),
            array(idx_type, "cc_left", self.left, int_fmt),
            array(idx_type, "cc_right", self.right, int_fmt),
            "/* raw window features -> model input. offset = NULL when not calibrated. */",
            "static inline void cc_prepare(const float *raw, const float *offset, float *out)",
            "{",
            "    for (int i = 0; i < CC_N_FEATURES; i++) {",
            "        float v = isnan(raw[i]) ? 0.0f : raw[i];",
            "        if (cc_log_feature[i]) {",
            "            v = logf(fmaxf(v, 0.0f) + 1e-6f);",
            "        }",
            "        if (offset) {",
            "            v -= offset[i];",
            "        }",
            "        out[i] = v;",
            "    }",
            "}",
            "",
            "/* model input -> people estimate (not yet clipped / smoothed) */",
            "static inline float cc_predict(const float *x)",
            "{",
            "    float total = 0.0f;",
            "    for (int t = 0; t < CC_N_TREES; t++) {",
            "        int n = cc_roots[t];",
            "        while (cc_feature[n] >= 0) {",
            "            n = (x[cc_feature[n]] <= cc_value[n]) ? cc_left[n] : cc_right[n];",
            "        }",
            "        total += cc_value[n];",
            "    }",
            "    return total;",
            "}",
            "",
            "#endif /* CC_COUNTER_MODEL_H */",
            "",
        ]
        with open(path, "w") as f:
            f.write("\n".join(text))
        return os.path.getsize(path)

    def flash_bytes(self):
        """Approximate flash used by the node arrays in the C header."""
        idx_bytes = 2 if self.n_nodes < 32767 else 4
        per_node = 1 + 4 + 2 * idx_bytes                    # feature + value + left + right
        return self.n_nodes * per_node + len(self.roots) * idx_bytes


# ===============================================================
# Streaming counter (what the ESP32 runs)
# ===============================================================
class StreamingCounter:
    """Feed packets one by one with push(t_seconds, amplitude_52).

    Every 2.5 s (after the warm-up) push() returns a result dict with the window
    features and, if a model is loaded, the raw / smoothed estimate and the count.
    Packets must arrive in time order; late packets are dropped."""

    def __init__(self, model=None, offset=None):
        self.model = model
        self.offset = None if offset is None else np.asarray(offset, dtype=np.float64)

        # 50 Hz binning
        self.t0 = None
        self.cur_bin = -1
        self.acc_sum = None
        self.acc_n = 0
        self.last_valid_bin = -1
        self.last_valid_x = None

        # causal baseline + window buffer
        self.baseline = None
        self.z_buf = deque(maxlen=WIN)
        self.v_buf = deque(maxlen=WIN)
        self.n_bins = 0

        # output smoothing + calibration
        self.recent = deque(maxlen=SMOOTH_N)
        self.cal_feats = None
        self.cal_target = 0

    # ---------------------------------------------------------------
    # Calibration: leave the room empty, call begin_calibration(), keep pushing
    # ---------------------------------------------------------------
    def begin_calibration(self, n_windows=N_CAL_WINDOWS):
        self.cal_feats = []
        self.cal_target = n_windows

    @property
    def calibrating(self):
        return self.cal_feats is not None

    # ---------------------------------------------------------------
    # Packet input
    # ---------------------------------------------------------------
    def push(self, t, amplitude):
        x = gain_normalise(np.asarray(amplitude, dtype=np.float32)[None])[0]
        if self.t0 is None:
            self.t0 = float(t)
        b = int(math.floor((float(t) - self.t0) * FS))

        results = []
        if b < self.cur_bin:
            return results                                  # late packet

        if b > self.cur_bin:
            if self.acc_n:
                results += self._close_bin(self.cur_bin, self.acc_sum / self.acc_n)
            self.cur_bin = b
            self.acc_sum = np.zeros(N_TONES, dtype=np.float64)
            self.acc_n = 0

        self.acc_sum += x
        self.acc_n += 1
        return results

    def flush(self):
        """Close the last open bin (end of a recording)."""
        results = []
        if self.acc_n:
            results = self._close_bin(self.cur_bin, self.acc_sum / self.acc_n)
            self.acc_n = 0
        return results

    # ---------------------------------------------------------------
    # Internals
    # ---------------------------------------------------------------
    def _close_bin(self, k, mean):
        x = mean.astype(np.float32)
        out = []

        if self.last_valid_bin >= 0:
            gap = k - self.last_valid_bin - 1
            if 0 < gap <= MAX_GAP_BINS:
                left = self.last_valid_bin
                for j in range(left + 1, k):
                    w = (j - left) / (k - left)
                    xi = (self.last_valid_x * (1 - w) + x * w).astype(np.float32)
                    out += self._add_bin(xi, True)
            else:
                for _ in range(gap):
                    out += self._add_bin(None, False)

        self.last_valid_bin, self.last_valid_x = k, x
        out += self._add_bin(x, True)
        return out

    def _add_bin(self, x, valid):
        if valid:
            xd = x.astype(np.float64)
            if self.baseline is None:
                self.baseline = xd.copy()
            else:
                self.baseline = self.baseline + ALPHA * (xd - self.baseline)
            with np.errstate(invalid="ignore", divide="ignore"):
                z = np.where(self.baseline > EPS, xd / self.baseline - 1.0, 0.0)
            z = z.astype(np.float32)
        else:
            z = np.full(N_TONES, np.nan, dtype=np.float32)

        self.z_buf.append(z)
        self.v_buf.append(valid)
        self.n_bins += 1

        a = self.n_bins - WIN
        if a < 0 or a % HOP or a < WARMUP_BINS:
            return []
        if np.mean(self.v_buf) < 1 - MAX_MISSING:
            return []

        feats = window_features(np.array(self.z_buf))
        return [self._decide(a, feats)]

    def _decide(self, start_bin, feats):
        res = {"t_start_s": start_bin / FS, "t_end_s": (start_bin + WIN) / FS,
               "features": feats}

        if self.calibrating:
            self.cal_feats.append(feats)
            res["status"] = "calibrating"
            if len(self.cal_feats) >= self.cal_target:
                self.offset = calibration_offset(np.array(self.cal_feats))
                self.cal_feats = None
                self.recent.clear()
                res["status"] = "calibrated"
            return res

        if self.model is None:
            return res

        x = prepare_features(feats[None], self.offset)
        raw = float(np.clip(self.model.predict(x)[0], 0, self.model.max_count))
        self.recent.append(raw)
        smoothed = float(np.median(self.recent))
        res.update(status="ok", raw=raw, smoothed=smoothed,
                   count=int(to_count(smoothed, self.model.max_count)))
        return res


# ===============================================================
# Model bundle (what goes to the device / the friend's demo)
# ===============================================================
def save_bundle(model_dir, tree_model, card, booster=None):
    """Writes trees.json, model_card.json, counter_model.h (+ model.txt if booster)."""
    os.makedirs(model_dir, exist_ok=True)
    tree_model.save(os.path.join(model_dir, "trees.json"))

    note = f"{card.get('name', 'counter')} | calibration={card.get('calibration')} | " \
           f"counts 0-{tree_model.max_count} | {time.strftime('%Y-%m-%d %H:%M')}"
    header_bytes = tree_model.write_c_header(os.path.join(model_dir, "counter_model.h"), note)

    if booster is not None:
        booster.save_model(os.path.join(model_dir, "model.txt"))

    card = {**card, "settings": settings_dict(), "n_trees": len(tree_model.roots),
            "n_nodes": tree_model.n_nodes, "flash_bytes_est": tree_model.flash_bytes(),
            "header_file_bytes": header_bytes}
    with open(os.path.join(model_dir, "model_card.json"), "w") as f:
        json.dump(card, f, indent=1)
    return card


def load_bundle(model_dir):
    """Returns (TreeModel, card). Use with StreamingCounter(model)."""
    model = TreeModel.load(os.path.join(model_dir, "trees.json"))
    with open(os.path.join(model_dir, "model_card.json")) as f:
        card = json.load(f)
    if card["settings"] != settings_dict():
        raise ValueError("model_card settings differ from this esp32_counter.py")
    return model, card

In [ ]:
# 6. Device (causal) cues for mat80 / CountFi / WiCount: 10 base + 9 extra cues per window and view
sys.path.insert(0, "/kaggle/working")
import esp32_counter as ec

# Extra cues for "how many separate movers" (all cheap enough for an ESP32)
EXTRA_NAMES = [
    "pc2_frac",          # share of variance in the 2nd principal component
    "pc3_frac",          # ... and in the 3rd (several movers spread variance over components)
    "eig_entropy",       # entropy of the PCA spectrum = effective number of independent motions
    "doppler_centroid",  # mean temporal frequency of the motion (Hz)
    "doppler_spread",    # spread of the temporal frequencies (Hz)
    "psd_entropy",       # flatness of the temporal spectrum (0..1)
    "activity_cv",       # burstiness of motion over 0.2-s segments (one mover: bursty, many: steady)
    "tone_energy_cv",    # how unevenly motion is spread over the 52 tones
    "half_band_corr",    # correlation of motion in the lower vs upper 26 tones
]
LINK_NAMES = [
    "link_env_corr",     # mean correlation of the 9 links' motion envelopes (needs several receivers)
    "link_energy_cv",    # spread of motion energy over the 9 links
]


def extra_features(W):
    """W: (bins, 52) normalised window -> (extra cues, motion envelope over time)."""
    W = np.nan_to_num(W.astype(np.float32))
    Xc = W - W.mean(axis=0)

    # spread of the motion over principal components
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    eig_entropy = -(frac * np.log(frac + 1e-12)).sum()

    # temporal (Doppler) spectrum summed over tones, DC removed
    P = (np.abs(np.fft.rfft(Xc, axis=0)) ** 2).sum(axis=1)[1:]
    f = np.fft.rfftfreq(len(W), 1.0 / FS)[1:]
    p = P / (P.sum() + 1e-12)
    centroid = (p * f).sum()
    spread = np.sqrt((p * (f - centroid) ** 2).sum())
    psd_entropy = -(p * np.log(p + 1e-12)).sum() / np.log(len(p))

    # motion = absolute change between bins
    d = np.abs(np.diff(W, axis=0))
    env = d.mean(axis=1)
    seg = env[:len(env) // 10 * 10].reshape(-1, 10).mean(axis=1)
    activity_cv = seg.std() / (seg.mean() + 1e-9)
    tone_e = d.mean(axis=0)
    tone_energy_cv = tone_e.std() / (tone_e.mean() + 1e-9)

    half = W.shape[1] // 2
    lo, hi = d[:, :half].mean(axis=1), d[:, half:].mean(axis=1)
    half_band_corr = np.corrcoef(lo, hi)[0, 1] if lo.std() > 0 and hi.std() > 0 else 0.0

    cues = np.array([frac[1], frac[2], eig_entropy, centroid, spread, psd_entropy,
                     activity_cv, tone_energy_cv, half_band_corr], dtype=np.float32)
    return cues, env


def link_cues(envs):
    """Cross-link cues from the motion envelopes of all links of one sample."""
    E = np.array(envs)
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(E)
    energy = E.mean(axis=1)
    return np.array([np.nanmean(C[np.triu_indices(len(E), k=1)]),
                     energy.std() / (energy.mean() + 1e-9)], dtype=np.float32)


def causal_rows(path):
    """Same steps as the device: causal 30 s baseline, warm-up, 5 s windows every 2.5 s."""
    with np.load(path) as z:
        S, valid = z["series"], z["valid"]
        views = [str(v) for v in z["view_names"]]
        meta = json.loads(str(z["meta"]))

    rows = []
    for v, view in enumerate(views):
        X = S[v].astype(np.float32) + 1.0          # Phase 1 stored x / recording_mean - 1; the mean cancels
        Z = ec.causal_normalise(X, valid)
        for j, a in enumerate(ec.window_starts(valid)):
            W = Z[a:a + ec.WIN]
            extra, _ = extra_features(W)
            rows.append({"dataset": meta["dataset"], "recording": meta["recording"], "setup": meta["setup"],
                         "receiver": meta["receiver"], "count": int(meta["count"]), "view": view,
                         "window": j, "t_start_s": a / ec.FS,
                         **dict(zip(ec.FEATURE_NAMES, ec.window_features(W))),
                         **dict(zip(EXTRA_NAMES, extra))})
    return rows


npz_paths = sorted(glob.glob(os.path.join(OUT_ROOT, "*", "*.npz")))
print(len(npz_paths), "recordings")
all_rows = []
t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    for k, rows in enumerate(pool.imap_unordered(causal_rows, npz_paths)):
        all_rows += rows
        if (k + 1) % 20 == 0:
            print(f"{k + 1:4d} / {len(npz_paths)}  [{time.time() - t0:.0f} s]", flush=True)

streams = pd.DataFrame(all_rows)
streams.to_parquet("/kaggle/working/streams_features.parquet", index=False)
print(len(streams), "rows")
first_view = streams.groupby("dataset")["view"].transform("first")
display(streams[streams["view"] == first_view].groupby(["dataset", "count"])
        .agg(recordings=("recording", "nunique"), windows=("window", "size")))

In [ ]:
# 1. Setup and paths
import copy
import glob
import json
import os
import time
from multiprocessing import get_context

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from sklearn.metrics import f1_score

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))

roots = find_dirs(lambda folder, files: "annotation.csv" in files)
assert roots, "annotation.csv not found - is the WiMANS dataset attached?"
WIMANS_ROOT = roots[0]
ANNOT = os.path.join(WIMANS_ROOT, "annotation.csv")
AMP_DIR = os.path.join(WIMANS_ROOT, "wifi_csi", "amp")
if not os.path.isdir(AMP_DIR):
    hits = glob.glob(os.path.join(WIMANS_ROOT, "**", "amp"), recursive=True)
    AMP_DIR = hits[0]

OUT_ROOT = "/kaggle/working/shared_format"
SHARED = os.path.join(OUT_ROOT, "wimans")
RESULTS = "/kaggle/working/results/wimans"
os.makedirs(SHARED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# ---- shared-format settings (identical to the local-dataset notebooks) ----
FS = 50              # common sample rate (Hz)
MAX_GAP_BINS = 3     # gaps up to 60 ms are interpolated
MAX_MISSING = 0.20   # samples/windows with more missing bins are dropped
N_TONES = 52
WIN_S = 3.0          # one WiMANS sample = one window
HOP_S = 3.0
WIMANS_RATE = 1000   # nominal packets/s (amp files carry no timestamps)
BANDS_USED = ["5"]   # 2.4 GHz is white noise in WiMANS (see the Phase 0 signal probe)
N_BINS = int(WIN_S * FS)
N_WORKERS = os.cpu_count()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
print("WiMANS root:", WIMANS_ROOT)
print("amp dir    :", AMP_DIR, "  files:", len(os.listdir(AMP_DIR)))
print("device     :", DEVICE, "  cpus:", N_WORKERS)

In [ ]:
# 2. Phase 0 - labels
ann = pd.read_csv(ANNOT, dtype=str)
print(ann.shape)
print(list(ann.columns))

ann["count"] = ann["number_of_users"].astype(int)
ann["session"] = ann["label"].str.split("_").str[1]          # act_<session>_<sample>
ann["has_file"] = [os.path.exists(os.path.join(AMP_DIR, l + ".npy")) for l in ann["label"]]
print("annotation rows with an amp file:", ann["has_file"].sum(), "/", len(ann))

display(pd.crosstab([ann["environment"], ann["wifi_band"]], ann["count"], margins=True))

# how many sessions per environment/band, and are counts mixed inside a session?
per_session = ann.groupby("session").agg(env=("environment", "first"), band=("wifi_band", "first"),
                                         n=("label", "size"), counts=("count", "nunique"),
                                         envs=("environment", "nunique"), bands=("wifi_band", "nunique"))
print("sessions:", len(per_session), "| sessions spanning >1 env:", (per_session["envs"] > 1).sum(),
      "| >1 band:", (per_session["bands"] > 1).sum(), "| with >1 count:", (per_session["counts"] > 1).sum())
display(per_session.groupby(["env", "band"]).agg(sessions=("n", "size"), samples=("n", "sum")))

act_cols = [c for c in ann.columns if c.endswith("_activity")]
print("activities:", sorted(set(ann[act_cols].stack().dropna()) - {"nan"}))

In [ ]:
# 3. Shared-format helpers: sub-bands, 50 Hz grid, normalisation, count cues

# ---------------------------------------------------------------
# 20 MHz sub-bands of an 80 MHz channel (256-point FFT, natural order)
# ---------------------------------------------------------------
# Sub-band s covers tone indices -128+64s ... -65+64s. We keep 52 tones,
# skipping 6 at each edge. This automatically avoids DC (bins 0, 1, 255)
# and the guard band (bins 123-133) that carry no channel data.
def subband_bins(s):
    start = -128 + 64 * s
    tones = np.arange(start + 6, start + 6 + N_TONES)
    return np.mod(tones, 256)


SUBBANDS = [subband_bins(s) for s in range(4)]


# ---------------------------------------------------------------
# Normalisation
# ---------------------------------------------------------------
def gain_normalise(A):
    """Per packet: divide by the median over tones (removes AGC jumps)."""
    return A / (np.median(A, axis=1, keepdims=True) + 1e-6)


def recording_normalise(X):
    """Per recording + receiver: each tone relative to its mean -> fluctuation around 0."""
    mu = np.nanmean(X, axis=0, keepdims=True)
    dead = mu < 1e-3 * np.nanmedian(mu)                    # null tones, if any
    Y = X / (mu + 1e-6) - 1.0
    Y[:, dead[0]] = 0.0
    return Y


# ---------------------------------------------------------------
# Resampling onto the 50 Hz grid
# ---------------------------------------------------------------
def resample_to_grid(t, A):
    """t: packet times (s, sorted), A: (packets, tones).
    Returns X (bins, tones) with NaN rows for empty bins, and the valid mask."""
    idx = np.floor((t - t[0]) * FS).astype(np.int64)
    n_bins = int(idx[-1]) + 1

    uniq, first = np.unique(idx, return_index=True)        # idx is sorted
    sums = np.add.reduceat(A.astype(np.float64), first, axis=0)
    counts = np.diff(np.append(first, len(idx)))

    X = np.full((n_bins, A.shape[1]), np.nan, dtype=np.float32)
    X[uniq] = sums / counts[:, None]
    valid = np.zeros(n_bins, dtype=bool)
    valid[uniq] = True
    return X, valid


def fill_short_gaps(X, valid):
    """Linear interpolation over gaps of at most MAX_GAP_BINS; longer gaps stay NaN."""
    X, valid = X.copy(), valid.copy()
    missing = np.flatnonzero(~valid)
    if len(missing) == 0:
        return X, valid

    runs = np.split(missing, np.flatnonzero(np.diff(missing) > 1) + 1)
    for run in runs:
        left, right = run[0] - 1, run[-1] + 1
        if len(run) <= MAX_GAP_BINS and left >= 0 and right < len(X):
            w = ((run - left) / (right - left))[:, None]
            X[run] = X[left] * (1 - w) + X[right] * w
            valid[run] = True
    return X, valid


def to_shared_series(t, A):
    """Raw amplitude of ONE view -> normalised 50 Hz series (bins, 52) + valid mask."""
    A = gain_normalise(A.astype(np.float32))
    X, valid = resample_to_grid(t, A)
    X, valid = fill_short_gaps(X, valid)
    return recording_normalise(X), valid


# ---------------------------------------------------------------
# Count cues per window
# ---------------------------------------------------------------
FEATURE_NAMES = [
    "cv_median",       # median over tones of the fluctuation std
    "cv_p90",          # 90th percentile of the same
    "cv_p10",          # 10th percentile of the same
    "diff_energy",     # mean |x(t) - x(t-1)|  (motion energy)
    "pc1_frac",        # share of variance in the first principal component
    "n_pc_90",         # components needed for 90 % of the variance
    "tone_corr",       # mean correlation between tones
    "low_freq_frac",   # temporal power below 2 Hz (slow body motion)
    "high_freq_frac",  # temporal power above 5 Hz
    "mean_std",        # std of the tone-averaged series
]


def window_features(W):
    """W: (bins, 52) normalised window. Missing bins count as 'no change' (0)."""
    W = np.nan_to_num(W.astype(np.float32))
    sd = W.std(axis=0)
    diff_energy = np.abs(np.diff(W, axis=0)).mean()

    Xc = W - W.mean(axis=0)
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    n90 = np.searchsorted(np.cumsum(frac), 0.90) + 1

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(Xc.T)
    tone_corr = np.nanmean(C[np.triu_indices_from(C, k=1)])

    x = Xc.mean(axis=1)
    P = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / FS)
    p_total = P[f > 0].sum() + 1e-12
    low = P[(f > 0) & (f < 2.0)].sum() / p_total
    high = P[f > 5.0].sum() / p_total

    return np.array([np.median(sd), np.percentile(sd, 90), np.percentile(sd, 10),
                     diff_energy, frac[0], n90, tone_corr, low, high, x.std()],
                    dtype=np.float32)


# ---------------------------------------------------------------
# Windowing + saving one recording
# ---------------------------------------------------------------
def save_recording(rec):
    """rec: dict with series (views, bins, 52), valid (bins,), view_names, metadata."""
    S, valid = rec["series"], rec["valid"]
    win, hop = int(WIN_S * FS), int(HOP_S * FS)

    starts = [a for a in range(0, S.shape[1] - win + 1, hop)
              if valid[a:a + win].mean() >= 1 - MAX_MISSING]

    F = np.zeros((S.shape[0], len(starts), len(FEATURE_NAMES)), dtype=np.float32)
    for v in range(S.shape[0]):
        for j, a in enumerate(starts):
            F[v, j] = window_features(S[v, a:a + win])

    meta = {k: rec[k] for k in ["dataset", "recording", "group", "setup", "receiver", "count"]}
    meta["seconds"] = round(S.shape[1] / FS, 1)
    meta["missing_frac"] = round(float(1 - valid.mean()), 4)
    meta["n_windows"] = len(starts)

    out_dir = os.path.join(OUT_ROOT, rec["dataset"])
    os.makedirs(out_dir, exist_ok=True)
    np.savez_compressed(os.path.join(out_dir, rec["recording"] + ".npz"),
                        series=S.astype(np.float16), valid=valid,
                        starts=np.array(starts, dtype=np.int32), feats=F,
                        view_names=np.array(rec["view_names"]), meta=json.dumps(meta))
    return meta


def already_done(dataset, recording):
    return os.path.exists(os.path.join(OUT_ROOT, dataset, recording + ".npz"))


print("sub-band bins:", [(int(b[0]), int(b[-1])) for b in SUBBANDS])


# Map any number of subcarriers onto the 52-tone grid (linear interpolation over frequency)
def tone_interp_matrix(n_src, n_dst=N_TONES):
    src = np.linspace(0.0, 1.0, n_src)
    dst = np.linspace(0.0, 1.0, n_dst)
    W = np.zeros((n_dst, n_src), dtype=np.float32)
    for i, x in enumerate(dst):
        j = min(np.searchsorted(src, x, side="right") - 1, n_src - 2)
        w = (x - src[j]) / (src[j + 1] - src[j])
        W[i, j] = 1.0 - w
        W[i, j + 1] = w
    return W

In [ ]:
# 4. Phase 1 - convert every 5 GHz sample to the shared format (9 views x 150 bins x 52 tones)
W30 = tone_interp_matrix(30)
labels = ann.loc[ann["has_file"] & ann["wifi_band"].isin(BANDS_USED), "label"].tolist()
print("samples to convert:", len(labels), "bands:", BANDS_USED)


def convert_sample(label):
    A = np.load(os.path.join(AMP_DIR, label + ".npy")).astype(np.float32)   # (T, 3, 3, 30)
    t = np.arange(A.shape[0]) / WIMANS_RATE
    series = np.full((9, N_BINS, N_TONES), np.nan, dtype=np.float32)
    valid = np.zeros(N_BINS, dtype=bool)
    feats = np.zeros((9, len(FEATURE_NAMES)), dtype=np.float32)

    for v in range(9):
        a, b = divmod(v, 3)
        X, ok = to_shared_series(t, A[:, a, b, :] @ W30.T)
        n = min(len(X), N_BINS)
        series[v, :n] = X[:n]
        valid[:n] = ok[:n]
        feats[v] = window_features(series[v])
    return label, series.astype(np.float16), valid, feats


N = len(labels)
SERIES = np.zeros((N, 9, N_BINS, N_TONES), dtype=np.float16)       # ~0.8 GB for 5 GHz
VALID = np.zeros((N, N_BINS), dtype=bool)
FEATS = np.zeros((N, 9, len(FEATURE_NAMES)), dtype=np.float32)
pos = {l: i for i, l in enumerate(labels)}

t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    for k, (label, s, ok, f) in enumerate(pool.imap_unordered(convert_sample, labels, chunksize=16)):
        i = pos[label]
        SERIES[i], VALID[i], FEATS[i] = s, ok, f
        if k % 1000 == 0:
            print(f"{k:6d} / {N}   [{time.time() - t0:.0f} s]", flush=True)

meta = ann.set_index("label").loc[labels].reset_index()
meta["valid_frac"] = VALID.mean(1)
np.save(os.path.join(SHARED, "series.npy"), SERIES)
np.save(os.path.join(SHARED, "feats.npy"), FEATS)
meta.to_parquet(os.path.join(SHARED, "meta.parquet"), index=False)
with open(os.path.join(SHARED, "format.json"), "w") as f:
    json.dump({"fs_hz": FS, "win_s": WIN_S, "n_tones": N_TONES, "views": "tx*3+rx (9 links)",
               "tone_mapping": "30 Intel-5300 subcarriers linearly interpolated to 52",
               "feature_names": FEATURE_NAMES}, f, indent=1)
print(f"done in {time.time() - t0:.0f} s; samples with >= 80 % valid bins:",
      int((meta["valid_frac"] >= 1 - MAX_MISSING).sum()), "/", N)

In [ ]:
# 6. Phase 2a - extra cues for "how many separate movers" (all cheap enough for an ESP32)
# Extra cues for "how many separate movers" (all cheap enough for an ESP32)
EXTRA_NAMES = [
    "pc2_frac",          # share of variance in the 2nd principal component
    "pc3_frac",          # ... and in the 3rd (several movers spread variance over components)
    "eig_entropy",       # entropy of the PCA spectrum = effective number of independent motions
    "doppler_centroid",  # mean temporal frequency of the motion (Hz)
    "doppler_spread",    # spread of the temporal frequencies (Hz)
    "psd_entropy",       # flatness of the temporal spectrum (0..1)
    "activity_cv",       # burstiness of motion over 0.2-s segments (one mover: bursty, many: steady)
    "tone_energy_cv",    # how unevenly motion is spread over the 52 tones
    "half_band_corr",    # correlation of motion in the lower vs upper 26 tones
]
LINK_NAMES = [
    "link_env_corr",     # mean correlation of the 9 links' motion envelopes (needs several receivers)
    "link_energy_cv",    # spread of motion energy over the 9 links
]


def extra_features(W):
    """W: (bins, 52) normalised window -> (extra cues, motion envelope over time)."""
    W = np.nan_to_num(W.astype(np.float32))
    Xc = W - W.mean(axis=0)

    # spread of the motion over principal components
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    eig_entropy = -(frac * np.log(frac + 1e-12)).sum()

    # temporal (Doppler) spectrum summed over tones, DC removed
    P = (np.abs(np.fft.rfft(Xc, axis=0)) ** 2).sum(axis=1)[1:]
    f = np.fft.rfftfreq(len(W), 1.0 / FS)[1:]
    p = P / (P.sum() + 1e-12)
    centroid = (p * f).sum()
    spread = np.sqrt((p * (f - centroid) ** 2).sum())
    psd_entropy = -(p * np.log(p + 1e-12)).sum() / np.log(len(p))

    # motion = absolute change between bins
    d = np.abs(np.diff(W, axis=0))
    env = d.mean(axis=1)
    seg = env[:len(env) // 10 * 10].reshape(-1, 10).mean(axis=1)
    activity_cv = seg.std() / (seg.mean() + 1e-9)
    tone_e = d.mean(axis=0)
    tone_energy_cv = tone_e.std() / (tone_e.mean() + 1e-9)

    half = W.shape[1] // 2
    lo, hi = d[:, :half].mean(axis=1), d[:, half:].mean(axis=1)
    half_band_corr = np.corrcoef(lo, hi)[0, 1] if lo.std() > 0 and hi.std() > 0 else 0.0

    cues = np.array([frac[1], frac[2], eig_entropy, centroid, spread, psd_entropy,
                     activity_cv, tone_energy_cv, half_band_corr], dtype=np.float32)
    return cues, env


def link_cues(envs):
    """Cross-link cues from the motion envelopes of all links of one sample."""
    E = np.array(envs)
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(E)
    energy = E.mean(axis=1)
    return np.array([np.nanmean(C[np.triu_indices(len(E), k=1)]),
                     energy.std() / (energy.mean() + 1e-9)], dtype=np.float32)


def sample_extra(i):
    cues, envs = [], []
    for v in range(9):
        c, e = extra_features(SERIES[i, v])
        cues.append(c)
        envs.append(e)
    return i, np.array(cues), link_cues(envs)


EXTRA = np.zeros((N, 9, len(EXTRA_NAMES)), dtype=np.float32)
LINK = np.zeros((N, len(LINK_NAMES)), dtype=np.float32)
t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    for i, cues, link in pool.imap_unordered(sample_extra, range(N), chunksize=64):
        EXTRA[i], LINK[i] = cues, link
print(f"extra cues for {N} samples in {time.time() - t0:.0f} s")
np.save(os.path.join(SHARED, "extra_feats.npy"), EXTRA)
np.save(os.path.join(SHARED, "link_feats.npy"), LINK)

# how each cue moves with the people count, per room (median over samples, mean over links)
cue = pd.DataFrame(EXTRA.mean(axis=1), columns=EXTRA_NAMES)
cue[LINK_NAMES] = LINK
cue["count"], cue["env"] = meta["count"].values, meta["environment"].values
names = EXTRA_NAMES + LINK_NAMES
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, names):
    for env, g in cue.groupby("env"):
        m = g.groupby("count")[col].median()
        ax.plot(m.index, m.values, marker="o", label=env)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("people")
axes.flat[0].legend(fontsize=7)
for ax in axes.flat[len(names):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "extra_cues_vs_count.png"), dpi=110)
plt.show()

spearman = pd.DataFrame({env: [g[c].corr(g["count"], method="spearman") for c in names]
                         for env, g in cue.groupby("env")}, index=names).round(3)
spearman.to_csv(os.path.join(RESULTS, "extra_cue_spearman.csv"))
display(spearman)

In [ ]:
# 11. WiMANS -> one feature row per (clip, link); session + clip number kept for the empty-room calibration
import shutil

EXPORT = "/kaggle/working"
LINK_NAMES_W = [f"tx{a}_rx{b}" for a in range(3) for b in range(3)]

keep = meta["valid_frac"].values >= 1 - MAX_MISSING
idx = np.flatnonzero(keep)
n_links = len(LINK_NAMES_W)

wim = pd.DataFrame({"clip": np.repeat(idx, n_links), "view": np.tile(LINK_NAMES_W, len(idx))})
wim["dataset"] = "wimans"
wim["recording"] = "wimans_" + meta["label"].values[wim["clip"]]
wim["setup"] = meta["environment"].values[wim["clip"]]
wim["session"] = meta["session"].values[wim["clip"]]
wim["sample_no"] = pd.to_numeric(meta["label"].str.split("_").str[2], errors="coerce").values[wim["clip"]]
wim["count"] = meta["count"].values[wim["clip"]].astype(int)
wim["window_s"] = WIN_S
wim[FEATURE_NAMES] = FEATS[idx].reshape(-1, len(FEATURE_NAMES))
wim[EXTRA_NAMES] = EXTRA[idx].reshape(-1, len(EXTRA_NAMES))
wim = wim.drop(columns="clip")

wim.to_parquet(os.path.join(EXPORT, "wimans_clip_features.parquet"), index=False)
print(f"WiMANS: {wim['recording'].nunique()} clips, {len(wim)} rows")
display(pd.crosstab(wim.drop_duplicates("recording")["setup"], wim.drop_duplicates("recording")["count"],
                    margins=True))

# ---- free memory + disk before CSI-Bench (the raw series are not needed any more) ----
del SERIES, VALID
shutil.rmtree("/kaggle/working/shared_format", ignore_errors=True)
shutil.rmtree("/kaggle/working/results", ignore_errors=True)

In [ ]:
# 1. Setup and task discovery
import copy
import glob
import json
import os
import re
import time
from multiprocessing import get_context

import h5py
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from sklearn.metrics import f1_score

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))

TASK_DIRS = find_dirs(lambda folder, files: os.path.exists(
    os.path.join(folder, "metadata", "sample_metadata.csv")), max_depth=7)
assert TASK_DIRS, "no <task>/metadata/sample_metadata.csv found - is CSI-Bench attached?"
for d in TASK_DIRS:
    print(d)

OUT_ROOT = "/kaggle/working/shared_format"
AUDIT = "/kaggle/working/csibench_audit"
SHARED = os.path.join(OUT_ROOT, "csibench_esp32")
SHARED_LOC = os.path.join(OUT_ROOT, "csibench_loc")
RESULTS = "/kaggle/working/results/csibench"
for folder in [AUDIT, SHARED, SHARED_LOC, RESULTS]:
    os.makedirs(folder, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
print("device:", DEVICE)

# ---- shared-format settings (identical to the local-dataset notebooks) ----
FS = 50              # common sample rate (Hz)
MAX_GAP_BINS = 3     # gaps up to 60 ms are interpolated
MAX_MISSING = 0.20   # samples/windows with more missing bins are dropped
N_TONES = 52
WIN_S = 5.0                   # CSI-Bench samples are 5 s (10 s for breathing)
HOP_S = 5.0
DEVICE_PATTERN = "esp32"      # case-insensitive match on the device name
ASSUMED_RATE_HZ = 100         # checked against the sample lengths in cell 3
N_WORKERS = os.cpu_count()

PATH_FIELDS = re.compile(r"user_(?P<user>[^/]+)/act_(?P<act>[^/]+)/env_(?P<env>[^/]+)/"
                         r"device_(?P<device>[^/]+)/session_(?P<session>[^/]+?)__freq(?P<freq>[^/]+?)\.h5$")


def read_amps(f):
    """CSI-Bench stores CSI_amps as (subcarriers, time, 1). Return (time, subcarriers)."""
    key = "CSI_amps" if "CSI_amps" in f else sorted(f.keys())[0]
    a = np.asarray(f[key], dtype=np.float32)
    if a.ndim == 3:
        a = a[:, :, 0]
    return a.T


def from_sub_folder(p):
    """Cut a metadata file_path down to the part starting at its first 'sub_*' folder."""
    parts = p.replace("\\", "/").split("/")
    for i, part in enumerate(parts):
        if part.startswith("sub_"):
            return "/".join(parts[i:])
    return p.lstrip("./")


def resolve_paths(task_dir, rel_paths, n_test=20):
    """Find which folder (the task folder or one of its parents) the file paths are relative to."""
    tail = [from_sub_folder(p) for p in rel_paths]
    base = task_dir
    while base.count("/") > 2:
        hits = sum(os.path.exists(os.path.join(base, t)) for t in tail[:n_test])
        if hits >= max(1, min(n_test, len(tail)) // 2):
            return [os.path.join(base, t) for t in tail], base
        base = os.path.dirname(base)
    print("  WARNING: could not resolve file paths for", task_dir)
    return [os.path.normpath(os.path.join(task_dir, p)) for p in rel_paths], None

In [ ]:
# 2. Phase 0 - metadata, labels, devices and splits per task
def task_name(d):
    parts = d.rstrip("/").split("/")
    return parts[-1] if parts[-2] != "Multitask" else "Multitask/" + parts[-1]


all_meta, audit = [], {}
for d in TASK_DIRS:
    name = task_name(d)
    md = pd.read_csv(os.path.join(d, "metadata", "sample_metadata.csv"))
    print("=" * 100)
    print(name, md.shape)
    print("columns:", list(md.columns))
    display(md.head(3))

    # fields encoded in the file path (user / activity / environment / device / session / freq)
    fp = md["file_path"].astype(str) if "file_path" in md.columns else pd.Series([""] * len(md))
    parsed = fp.str.extract(PATH_FIELDS)
    for c in parsed.columns:
        if c not in md.columns:
            md[c] = parsed[c]
    md["abs_path"], base = resolve_paths(d, fp.tolist())
    print("  files resolved under:", base)
    md["task"] = name

    # Localization labels are zone bitmasks: number of 1s = number of people
    if name == "Localization":
        md["count"] = md["label"].astype(str).str.count("1")
        print("  people count from label bitmask:")
        display(pd.crosstab(md["count"], md.get("is_multi_subject", "?"), margins=True))

    low_card = [c for c in md.columns if 1 < md[c].nunique() <= 40 and c not in ("abs_path", "file_path")]
    for c in low_card:
        print(f"  {c}: {md[c].value_counts().to_dict()}")

    splits = {}
    for sp in sorted(glob.glob(os.path.join(d, "splits", "*.json"))):
        ids = json.load(open(sp))
        splits[os.path.basename(sp)[:-5]] = len(ids)
    print("  splits:", splits)
    mapping = os.path.join(d, "metadata", "label_mapping.json")
    if os.path.exists(mapping):
        print("  label_mapping:", json.load(open(mapping)))

    audit[name] = {"rows": len(md), "columns": list(md.columns), "splits": splits, "files_base": base,
                   "value_counts": {c: md[c].astype(str).value_counts().to_dict() for c in low_card}}
    all_meta.append(md)

meta = pd.concat(all_meta, ignore_index=True)
if "device" not in meta.columns:
    meta["device"] = "?"
meta["is_esp32"] = meta["device"].astype(str).str.contains(DEVICE_PATTERN, case=False, na=False)
print("=" * 100)
display(pd.crosstab(meta["task"], meta["device"].fillna("?"), margins=True))
with open(os.path.join(AUDIT, "metadata_audit.json"), "w") as f:
    json.dump(audit, f, indent=1, default=str)

In [ ]:
# 6. Localization - layout check: are the 168 subcarriers really 3 devices x 56?
N_BINS = int(WIN_S * FS)                              # one CSI-Bench clip = one 5-s window
LOC_DEVICES = 3
LOC_SUB = 168

loc = meta[meta["task"] == "Localization"].drop_duplicates("abs_path").reset_index(drop=True)
loc["n_sub"] = loc["num_sub"].astype(int)
loc["count"] = loc["count"].astype(int)
print("Localization samples per room / subcarrier count / people:")
display(pd.crosstab([loc["environment"], loc["n_sub"]], loc["count"], margins=True))
display(pd.crosstab(loc["device"], loc["n_sub"]))


def mean_profile(paths):
    """Mean |amplitude| per subcarrier over several files."""
    acc = None
    for p in paths:
        with h5py.File(p, "r") as f:
            m = np.abs(read_amps(f)).mean(axis=0)
        acc = m if acc is None else acc + m
    return acc / len(paths)


profiles = {n: mean_profile(g["abs_path"].sample(min(60, len(g)), random_state=SEED).tolist())
            for n, g in loc.groupby("n_sub")}

fig, axes = plt.subplots(len(profiles), 1, figsize=(14, 3 * len(profiles)))
for ax, (n_sub, prof) in zip(np.atleast_1d(axes), profiles.items()):
    ax.plot(prof)
    for k in range(1, LOC_DEVICES):
        ax.axvline(k * n_sub / LOC_DEVICES, color="r", ls="--")
    ax.set_title(f"mean amplitude per subcarrier, {n_sub} subcarriers (red = assumed device borders)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "loc_subcarrier_profile.png"), dpi=110)
plt.show()

# if the 3 x 56 layout is right, each device block has its own null tones at the same places
blocks = profiles[LOC_SUB].reshape(LOC_DEVICES, -1)
print("correlation between the 3 device blocks' profiles:")
print(np.round(np.corrcoef(blocks), 3))

In [ ]:
# 7. Localization - 168-subcarrier samples -> 3 links x 250 bins x 52 tones + count cues
SUB_PER_DEV = LOC_SUB // LOC_DEVICES
loc_use = loc[loc["n_sub"] == LOC_SUB].reset_index(drop=True)
prof = blocks.mean(axis=0)
KEEP_LOC = prof > 0.02 * np.median(prof)              # drop null tones inside a device block
W_LOC = tone_interp_matrix(int(KEEP_LOC.sum()))
print(len(loc_use), "samples;", int(KEEP_LOC.sum()), "of", SUB_PER_DEV, "tones kept per device")

# Extra cues for "how many separate movers" (all cheap enough for an ESP32)
EXTRA_NAMES = [
    "pc2_frac",          # share of variance in the 2nd principal component
    "pc3_frac",          # ... and in the 3rd (several movers spread variance over components)
    "eig_entropy",       # entropy of the PCA spectrum = effective number of independent motions
    "doppler_centroid",  # mean temporal frequency of the motion (Hz)
    "doppler_spread",    # spread of the temporal frequencies (Hz)
    "psd_entropy",       # flatness of the temporal spectrum (0..1)
    "activity_cv",       # burstiness of motion over 0.2-s segments (one mover: bursty, many: steady)
    "tone_energy_cv",    # how unevenly motion is spread over the 52 tones
    "half_band_corr",    # correlation of motion in the lower vs upper 26 tones
]
LINK_NAMES = [
    "link_env_corr",     # mean correlation of the 9 links' motion envelopes (needs several receivers)
    "link_energy_cv",    # spread of motion energy over the 9 links
]


def extra_features(W):
    """W: (bins, 52) normalised window -> (extra cues, motion envelope over time)."""
    W = np.nan_to_num(W.astype(np.float32))
    Xc = W - W.mean(axis=0)

    # spread of the motion over principal components
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    eig_entropy = -(frac * np.log(frac + 1e-12)).sum()

    # temporal (Doppler) spectrum summed over tones, DC removed
    P = (np.abs(np.fft.rfft(Xc, axis=0)) ** 2).sum(axis=1)[1:]
    f = np.fft.rfftfreq(len(W), 1.0 / FS)[1:]
    p = P / (P.sum() + 1e-12)
    centroid = (p * f).sum()
    spread = np.sqrt((p * (f - centroid) ** 2).sum())
    psd_entropy = -(p * np.log(p + 1e-12)).sum() / np.log(len(p))

    # motion = absolute change between bins
    d = np.abs(np.diff(W, axis=0))
    env = d.mean(axis=1)
    seg = env[:len(env) // 10 * 10].reshape(-1, 10).mean(axis=1)
    activity_cv = seg.std() / (seg.mean() + 1e-9)
    tone_e = d.mean(axis=0)
    tone_energy_cv = tone_e.std() / (tone_e.mean() + 1e-9)

    half = W.shape[1] // 2
    lo, hi = d[:, :half].mean(axis=1), d[:, half:].mean(axis=1)
    half_band_corr = np.corrcoef(lo, hi)[0, 1] if lo.std() > 0 and hi.std() > 0 else 0.0

    cues = np.array([frac[1], frac[2], eig_entropy, centroid, spread, psd_entropy,
                     activity_cv, tone_energy_cv, half_band_corr], dtype=np.float32)
    return cues, env


def link_cues(envs):
    """Cross-link cues from the motion envelopes of all links of one sample."""
    E = np.array(envs)
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(E)
    energy = E.mean(axis=1)
    return np.array([np.nanmean(C[np.triu_indices(len(E), k=1)]),
                     energy.std() / (energy.mean() + 1e-9)], dtype=np.float32)


def convert_loc(path):
    try:
        with h5py.File(path, "r") as f:
            a = read_amps(f)                                     # (time, 168)
    except Exception:                                            # unreadable file
        return path, None, None, None, None, None

    t = np.arange(len(a)) / ASSUMED_RATE_HZ
    series = np.full((LOC_DEVICES, N_BINS, N_TONES), np.nan, dtype=np.float32)
    valid = np.zeros(N_BINS, dtype=bool)
    feats = np.zeros((LOC_DEVICES, len(FEATURE_NAMES)), dtype=np.float32)
    extra = np.zeros((LOC_DEVICES, len(EXTRA_NAMES)), dtype=np.float32)
    envs = []
    for v in range(LOC_DEVICES):
        block = a[:, v * SUB_PER_DEV:(v + 1) * SUB_PER_DEV][:, KEEP_LOC] @ W_LOC.T
        X, ok = to_shared_series(t, block)
        n = min(len(X), N_BINS)
        series[v, :n] = X[:n]
        valid[:n] = ok[:n]
        feats[v] = window_features(series[v])
        extra[v], e = extra_features(series[v])
        envs.append(e)
    return path, series.astype(np.float16), valid, feats, extra, link_cues(envs)


NL = len(loc_use)
LOC_SERIES = np.zeros((NL, LOC_DEVICES, N_BINS, N_TONES), dtype=np.float16)
LOC_VALID = np.zeros((NL, N_BINS), dtype=bool)
LOC_FEATS = np.zeros((NL, LOC_DEVICES, len(FEATURE_NAMES)), dtype=np.float32)
LOC_EXTRA = np.zeros((NL, LOC_DEVICES, len(EXTRA_NAMES)), dtype=np.float32)
LOC_LINK = np.zeros((NL, len(LINK_NAMES)), dtype=np.float32)
LOC_OK = np.zeros(NL, dtype=bool)
pos = {p: i for i, p in enumerate(loc_use["abs_path"])}

t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    results_iter = pool.imap_unordered(convert_loc, loc_use["abs_path"].tolist(), chunksize=32)
    for k, (p, s, v, f, e, l) in enumerate(results_iter):
        if s is not None:
            i = pos[p]
            LOC_SERIES[i], LOC_VALID[i], LOC_FEATS[i] = s, v, f
            LOC_EXTRA[i], LOC_LINK[i], LOC_OK[i] = e, l, True
        if k % 2000 == 0:
            print(f"{k:6d} / {NL}   [{time.time() - t0:.0f} s]", flush=True)

loc_use["converted"] = LOC_OK
loc_use["valid_frac"] = LOC_VALID.mean(1)
np.save(os.path.join(SHARED_LOC, "series.npy"), LOC_SERIES)
np.save(os.path.join(SHARED_LOC, "feats.npy"), LOC_FEATS)
np.save(os.path.join(SHARED_LOC, "extra_feats.npy"), LOC_EXTRA)
np.save(os.path.join(SHARED_LOC, "link_feats.npy"), LOC_LINK)
out_meta = loc_use.rename(columns={"abs_path": "source_path"})
obj_cols = out_meta.select_dtypes("object").columns
out_meta[obj_cols] = out_meta[obj_cols].astype(str)                  # mixed types break parquet
out_meta.to_parquet(os.path.join(SHARED_LOC, "meta.parquet"), index=False)
with open(os.path.join(SHARED_LOC, "format.json"), "w") as f:
    json.dump({"fs_hz": FS, "win_s": WIN_S, "n_tones": N_TONES, "assumed_rate_hz": ASSUMED_RATE_HZ,
               "views": "3 device links (168 = 3 x 56 subcarriers)", "tones_kept_per_device": int(KEEP_LOC.sum()),
               "feature_names": FEATURE_NAMES, "extra_names": EXTRA_NAMES, "link_names": LINK_NAMES}, f, indent=1)
print(f"converted {LOC_OK.sum()} / {NL} in {time.time() - t0:.0f} s;",
      "with >= 80 % valid bins:", int((loc_use["valid_frac"] >= 1 - MAX_MISSING).sum()))

In [ ]:
# 9. CSI-Bench Localization -> one feature row per (clip, device link)
keep = LOC_OK & (loc_use["valid_frac"].values >= 1 - MAX_MISSING)
idx = np.flatnonzero(keep)
LINK_NAMES_C = [f"dev{v}" for v in range(LOC_DEVICES)]

cb = pd.DataFrame({"clip": np.repeat(idx, LOC_DEVICES), "view": np.tile(LINK_NAMES_C, len(idx))})
cb["dataset"] = "csibench"
cb["recording"] = "csibench_" + pd.Series(idx).astype(str).str.zfill(6).repeat(LOC_DEVICES).values
cb["setup"] = loc_use["environment"].astype(str).values[cb["clip"]]
session_col = "session" if "session" in loc_use.columns else "abs_path"   # parsed from the file path
cb["session"] = loc_use[session_col].astype(str).values[cb["clip"]]
cb["sample_no"] = cb["clip"]
cb["count"] = loc_use["count"].values[cb["clip"]].astype(int)
cb["window_s"] = WIN_S
cb[FEATURE_NAMES] = LOC_FEATS[idx].reshape(-1, len(FEATURE_NAMES))
cb[EXTRA_NAMES] = LOC_EXTRA[idx].reshape(-1, len(EXTRA_NAMES))
cb = cb.drop(columns="clip")

cb.to_parquet(os.path.join(EXPORT, "csibench_clip_features.parquet"), index=False)
print(f"CSI-Bench: {cb['recording'].nunique()} clips, {len(cb)} rows")
display(pd.crosstab(cb.drop_duplicates("recording")["setup"], cb.drop_duplicates("recording")["count"],
                    margins=True))

In [ ]:
# 10. Keep only the two feature tables in the output (small download)
for name in ["shared_format", "csibench_audit", "results"]:
    shutil.rmtree(os.path.join(EXPORT, name), ignore_errors=True)

for name in sorted(os.listdir(EXPORT)):
    path = os.path.join(EXPORT, name)
    if os.path.isfile(path):
        print(f"{name:40s} {os.path.getsize(path) / 1e6:8.2f} MB")